In [21]:
df = pd.read_csv('/content/swiggy_all_menus_india (1).csv')

In [22]:
print("Dataset loaded successfully!")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

Dataset loaded successfully!
Number of rows: 197430
Number of columns: 9


In [23]:
df.head()

,State,City,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count
0,Karnataka,Bengaluru,Anand Sweets & Savouries,Rajarajeshwari Nagar,Snack,Butter Murukku-200gm,133.9,0.0,0
1,Karnataka,Bengaluru,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25
2,Karnataka,Bengaluru,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48
3,Karnataka,Bengaluru,Srinidhi Sagar Deluxe,Kengeri,Recommended,Kesari Bath,65.0,4.6,65
4,Karnataka,Bengaluru,Srinidhi Sagar Deluxe,Kengeri,Recommended,Mix Raitha,130.0,0.0,0


In [ ]:
# ============================================================
# 🍔 SWIGGY FOOD MENU CHATBOT
# Google Colab + Pandas + Groq
# ============================================================

# 1. Install Groq
!pip -q install groq


# 2. Import libraries

import pandas as pd
from groq import Groq
from google.colab import drive


# 3. Mount Google Drive

drive.mount('/content/drive')


# 4. Load the Swiggy dataset

file_path = "/content/swiggy_all_menus_india (1).csv"

df = pd.read_csv(file_path)

print("✅ Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())


# 5. Clean the dataset

df = df.dropna(
    subset=["Restaurant Name", "Dish Name"]
)

df["Dish_Search"] = (
    df["Dish Name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["Restaurant_Search"] = (
    df["Restaurant Name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["City_Search"] = (
    df["City"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["Category_Search"] = (
    df["Category"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print("\n✅ Dataset cleaned!")


# ============================================================
# 6. GROQ API KEY
# ============================================================

# ⚠️ Replace the text below with YOUR NEW Groq API key

GROQ_API_KEY = "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"


# Create Groq client

client = Groq(
    api_key=GROQ_API_KEY
)

print("✅ Groq connected successfully!")


# ============================================================
# 7. SEARCH DATASET
# ============================================================

def search_dataset(query):

    query = query.lower().strip()

    result = df[
        df["Dish_Search"].str.contains(
            query,
            na=False,
            regex=False
        )
        |
        df["Restaurant_Search"].str.contains(
            query,
            na=False,
            regex=False
        )
        |
        df["City_Search"].str.contains(
            query,
            na=False,
            regex=False
        )
        |
        df["Category_Search"].str.contains(
            query,
            na=False,
            regex=False
        )
    ]

    return result


# ============================================================
# 8. FIND RELEVANT RESULTS
# ============================================================

def find_results(question):

    question = question.lower().strip()

    # Try complete question

    result = search_dataset(question)

    if len(result) > 0:
        return result

    # Split question into words

    words = question.split()

    stop_words = {
        "what", "is", "the", "a", "an",
        "how", "much", "does", "do",
        "are", "there", "in", "at",
        "of", "for", "me", "show",
        "give", "tell", "which",
        "restaurant", "restaurants",
        "dish", "dishes", "food",
        "available", "please",
        "can", "you", "i", "want",
        "to", "has", "have",
        "with", "and", "on"
    }

    useful_words = [
        word
        for word in words
        if word not in stop_words
        and len(word) > 2
    ]

    results = []

    for word in useful_words:

        temp = search_dataset(word)

        if len(temp) > 0:
            results.append(temp)

    if len(results) > 0:

        combined = pd.concat(results)

        combined = combined.drop_duplicates()

        return combined

    return pd.DataFrame()


# ============================================================
# 9. PREPARE DATA FOR GROQ
# ============================================================

def create_context(result):

    if result.empty:
        return ""

    # Send only 20 results to Groq

    result = result.head(20)

    columns = [
        "State",
        "City",
        "Restaurant Name",
        "Location",
        "Category",
        "Dish Name",
        "Price (INR)",
        "Rating",
        "Rating Count"
    ]

    available_columns = [
        column
        for column in columns
        if column in result.columns
    ]

    context = result[
        available_columns
    ].to_string(index=False)

    return context


# ============================================================
# 10. ASK GROQ
# ============================================================

def ask_groq(question, context):

    prompt = f"""
You are a food menu chatbot.

You MUST answer ONLY using the dataset
information provided below.

DATASET INFORMATION:

{context}

USER QUESTION:

{question}

RULES:

1. Use ONLY the provided dataset.

2. Do NOT use outside knowledge.

3. Do NOT invent restaurants.

4. Do NOT invent dishes.

5. Do NOT invent prices.

6. Do NOT invent ratings.

7. Do NOT invent locations.

8. If the answer cannot be found in
   the provided dataset, reply:

Content not found in the dataset.

9. Keep the answer short and clear.

10. If multiple results are available,
show the most relevant ones.
"""

    response = client.chat.completions.create(

        model="llama-3.1-8b-instant",

        messages=[
            {
                "role": "system",
                "content":
                "You are a strict food menu chatbot. "
                "Only answer using the supplied dataset."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0
    )

    return response.choices[0].message.content


# ============================================================
# 11. CHATBOT
# ============================================================

def chatbot(question):

    question = question.strip()

    if question == "":
        return "Please enter a question."

    # Search dataset

    result = find_results(question)

    # Nothing found

    if result.empty:
        return "Content not found in the dataset."

    # Prepare dataset information

    context = create_context(result)

    # Ask Groq

    answer = ask_groq(
        question,
        context
    )

    return answer


# ============================================================
# 12. START CHATBOT
# ============================================================

print("\n" + "=" * 55)

print("🍔 SWIGGY FOOD MENU CHATBOT")

print("=" * 55)

print("""
You can ask about:

🍕 Dishes
🍽️ Restaurants
💰 Prices
⭐ Ratings
📍 Cities
🍴 Food categories

Type 'exit' to stop.
""")


# ============================================================
# 13. CHAT LOOP
# ============================================================

while True:

    question = input("\nYou: ")

    if question.lower().strip() == "exit":

        print("\n🤖 Chatbot: Goodbye! 👋")
        break

    answer = chatbot(question)

    print("\n🤖 Chatbot:", answer)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset loaded successfully!
Rows: 197430
Columns: 9

Columns:
['State', 'City', 'Restaurant Name', 'Location', 'Category', 'Dish Name', 'Price (INR)', 'Rating', 'Rating Count']

✅ Dataset cleaned!
✅ Groq connected successfully!

🍔 SWIGGY FOOD MENU CHATBOT

You can ask about:

🍕 Dishes
🍽️ Restaurants
💰 Prices
⭐ Ratings
📍 Cities
🍴 Food categories

Type 'exit' to stop.


You: Dishes

🤖 Chatbot: I'm a food menu chatbot. What dish would you like to know about?

You: in the range 200

🤖 Chatbot: You are looking for dishes in the price range of 200 INR. 

Here are the results:

1. Crispy Veggie Burger at McDonald's - 200.00 INR
2. Ginger Orange Cooler at Nandhana Palace - 135.00 INR
